# 패스트볼(FF/SI/FC) Shape 클러스터링

## 목적
패스트볼 3종(포심 FF, 싱커 SI, 커터 FC)을 구종 라벨이 아니라 실제 구속·무브먼트·회전 특성으로 재표현하여 GMM 클러스터링으로 자연스러운 그룹 구조를 확인한다.

## 주요 발견: 손잡이(투수 좌/우완) 아티팩트
`pfx_x`, `release_pos_x`, `spin_axis`는 포수 시점 좌표라 우투수/좌투수 간 부호가 반대로 나타난다.
이를 그대로 쓰면 GMM이 "구종 차이"가 아니라 "손잡이 차이"를 학습해버려서, 최초 k=2 클러스터링 결과는 손잡이(R/L)와 거의 완벽히 일치했다.
→ 세 변수를 모두 암사이드(arm-side) 기준으로 부호 통일한 뒤 재분석.

## 최종 결과
- 실루엣 계수 기준으로는 k=2가 근소하게 높지만, 이는 "커터 vs 포심·싱커"만 가르는 거친 구분이다.
- k=3에서 포심/싱커/커터가 실제 구속·무브먼트 특성 차이로 명확히 분리되며, 손잡이도 각 클러스터에 고르게 섞여 있다 (아티팩트 해소 확인).
- 2021~2025년 5개 시즌 모두 k=3에서 일관된 결과(실루엣 0.16~0.18)를 보였고, 5-fold 교차검증에서도 out-of-fold 실루엣이 학습 실루엣과 거의 일치해 안정성이 확인되었다.

## 결론
FF/SI/FC는 라벨 그대로도 실제 shape 공간에서 뚜렷이 구분되는 3개 그룹으로 재현되며, 이는 연도·좌우완에 걸쳐 안정적이다.

In [ ]:
import pandas as pd
import glob

# data/raw 폴더 안의 모든 statcast csv 파일 찾기
files = glob.glob("../data/raw/statcast_*.csv")
print(f"파일 개수: {len(files)}")
print(files[:5])  # 처음 5개만 확인

In [ ]:
# 파일 하나 불러와서 구조 확인
df_test = pd.read_csv(files[0])
print(df_test.shape)
print(df_test['pitch_type'].value_counts())

In [ ]:
# 모든 파일 합치기
df_list = [pd.read_csv(f) for f in files]
df_all = pd.concat(df_list, ignore_index=True)
print(df_all.shape)

# 패스트볼 3종만 필터링
target_pitches = ['FF', 'SI', 'FC']
df_fb = df_all[df_all['pitch_type'].isin(target_pitches)].copy()
print(df_fb.shape)
print(df_fb['pitch_type'].value_counts())
print(df_fb['game_year'].value_counts().sort_index())

In [ ]:
# 클러스터링에 쓸 피처 선택
features = ['release_speed', 'pfx_x', 'pfx_z', 'release_spin_rate',
            'spin_axis', 'release_pos_x', 'release_pos_z', 'release_extension']

# 결측치 확인
print(df_fb[features].isna().sum())

# 결측치 있는 행 제거
df_clean = df_fb.dropna(subset=features).copy()
print(f"\n결측 제거 전: {len(df_fb)}, 제거 후: {len(df_clean)}")

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df_clean[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)
print(X_scaled[:3])  # 스케일링된 값 미리보기

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
import numpy as np

# 계산 시간 절약을 위해 5만 개 랜덤 샘플링
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), size=50000, replace=False)
X_sample = X_scaled[sample_idx]

# k=2부터 6까지 돌려보면서 실루엣 계수 확인
results = {}
for k in range(2, 7):
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=1)
    labels = gmm.fit_predict(X_sample)
    score = silhouette_score(X_sample, labels)
    results[k] = score
    print(f"k={k}: silhouette={score:.4f}")

In [ ]:
years = sorted(df_clean['game_year'].unique())
year_results = {}

for year in years:
    df_year = df_clean[df_clean['game_year'] == year]
    X_year = scaler.transform(df_year[features].values)
    
    # 연도별 표본 2만개 샘플링 (속도 위해)
    np.random.seed(42)
    idx = np.random.choice(len(X_year), size=min(20000, len(X_year)), replace=False)
    X_year_sample = X_year[idx]
    
    scores = {}
    for k in range(2, 7):
        gmm = GaussianMixture(n_components=k, random_state=42, n_init=1)
        labels = gmm.fit_predict(X_year_sample)
        score = silhouette_score(X_year_sample, labels)
        scores[k] = score
    
    year_results[year] = scores
    best_k = max(scores, key=scores.get)
    print(f"{year}년 - best k={best_k} (silhouette={scores[best_k]:.4f})")
    print(f"   전체: {scores}")

In [ ]:
from sklearn.model_selection import KFold

cv_results = {}

for year in years:
    df_year = df_clean[df_clean['game_year'] == year]
    X_year = scaler.transform(df_year[features].values)
    
    np.random.seed(42)
    idx = np.random.choice(len(X_year), size=min(20000, len(X_year)), replace=False)
    X_year_sample = X_year[idx]
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []
    
    for train_idx, test_idx in kf.split(X_year_sample):
        X_train, X_test = X_year_sample[train_idx], X_year_sample[test_idx]
        
        gmm = GaussianMixture(n_components=2, random_state=42, n_init=1)
        gmm.fit(X_train)
        test_labels = gmm.predict(X_test)
        
        # 테스트 fold에 클러스터가 1개만 나오면 실루엣 계산 불가하니 예외처리
        if len(set(test_labels)) > 1:
            score = silhouette_score(X_test, test_labels)
            fold_scores.append(score)
    
    cv_results[year] = fold_scores
    print(f"{year}년 - CV 평균 실루엣: {np.mean(fold_scores):.4f} (fold별: {[f'{s:.3f}' for s in fold_scores]})")

In [ ]:
# k=2로 최종 클러스터링 (전체 샘플 기준)
gmm_final = GaussianMixture(n_components=2, random_state=42, n_init=1)
cluster_labels = gmm_final.fit_predict(X_sample)

# 샘플에 해당하는 원본 데이터 가져오기
df_sample = df_clean.iloc[sample_idx].copy()
df_sample['cluster'] = cluster_labels

# 클러스터별 구종 분포
print("클러스터별 구종 분포:")
print(pd.crosstab(df_sample['cluster'], df_sample['pitch_type']))

print("\n클러스터별 평균 특성:")
print(df_sample.groupby('cluster')[features].mean().round(2))

In [ ]:
# p_throws(투수 손잡이)가 클러스터랑 얼마나 겹치는지 확인
df_sample['p_throws'] = df_clean.iloc[sample_idx]['p_throws'].values
print(pd.crosstab(df_sample['cluster'], df_sample['p_throws']))

In [ ]:
# 손잡이 기준 좌우 부호 통일 (암사이드 +)
hand_factor = df_clean['p_throws'].map({'R': -1, 'L': 1})

df_clean['pfx_x_arm'] = df_clean['pfx_x'] * hand_factor
df_clean['release_pos_x_arm'] = df_clean['release_pos_x'] * hand_factor

# pfx_x, release_pos_x 대신 arm-side 통일 버전 사용
features_v2 = ['release_speed', 'pfx_x_arm', 'pfx_z', 'release_spin_rate',
               'spin_axis', 'release_pos_x_arm', 'release_pos_z', 'release_extension']

X2 = df_clean[features_v2].values
scaler2 = StandardScaler()
X2_scaled = scaler2.fit_transform(X2)

np.random.seed(42)
sample_idx2 = np.random.choice(len(X2_scaled), size=50000, replace=False)
X2_sample = X2_scaled[sample_idx2]

for k in range(2, 7):
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=1)
    labels = gmm.fit_predict(X2_sample)
    score = silhouette_score(X2_sample, labels)
    print(f"k={k}: silhouette={score:.4f}")

In [ ]:
# k=4로 최종 클러스터링 (부호 수정 버전)
gmm_final_v2 = GaussianMixture(n_components=4, random_state=42, n_init=1)
cluster_labels_v2 = gmm_final_v2.fit_predict(X2_sample)

df_sample2 = df_clean.iloc[sample_idx2].copy()
df_sample2['cluster'] = cluster_labels_v2

# 손잡이랑 안 겹치는지 확인 (이게 핵심 체크)
print("클러스터별 손잡이 분포:")
print(pd.crosstab(df_sample2['cluster'], df_sample2['p_throws']))

# 구종 분포 확인 (이게 우리가 원하는 결과)
print("\n클러스터별 구종 분포:")
print(pd.crosstab(df_sample2['cluster'], df_sample2['pitch_type']))

# 클러스터별 평균 특성
print("\n클러스터별 평균 특성:")
print(df_sample2.groupby('cluster')[features_v2].mean().round(2))

In [ ]:
# spin_axis도 손잡이 기준으로 통일 (좌완은 360도에서 빼서 미러링)
df_clean['spin_axis_arm'] = np.where(
    df_clean['p_throws'] == 'L',
    360 - df_clean['spin_axis'],
    df_clean['spin_axis']
)

features_v3 = ['release_speed', 'pfx_x_arm', 'pfx_z', 'release_spin_rate',
               'spin_axis_arm', 'release_pos_x_arm', 'release_pos_z', 'release_extension']

X3 = df_clean[features_v3].values
scaler3 = StandardScaler()
X3_scaled = scaler3.fit_transform(X3)

np.random.seed(42)
sample_idx3 = np.random.choice(len(X3_scaled), size=50000, replace=False)
X3_sample = X3_scaled[sample_idx3]

for k in range(2, 7):
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=1)
    labels = gmm.fit_predict(X3_sample)
    score = silhouette_score(X3_sample, labels)
    print(f"k={k}: silhouette={score:.4f}")

In [ ]:
# k=2로 클러스터링 (spin_axis까지 수정한 버전)
gmm_v3 = GaussianMixture(n_components=2, random_state=42, n_init=1)
cluster_labels_v3 = gmm_v3.fit_predict(X3_sample)

df_sample3 = df_clean.iloc[sample_idx3].copy()
df_sample3['cluster'] = cluster_labels_v3

print("클러스터별 손잡이 분포:")
print(pd.crosstab(df_sample3['cluster'], df_sample3['p_throws']))

print("\n클러스터별 구종 분포:")
print(pd.crosstab(df_sample3['cluster'], df_sample3['pitch_type']))

print("\n클러스터별 평균 특성:")
print(df_sample3.groupby('cluster')[features_v3].mean().round(2))

In [ ]:
# k=3으로 클러스터링 (spin_axis까지 수정한 버전)
gmm_k3 = GaussianMixture(n_components=3, random_state=42, n_init=1)
cluster_labels_k3 = gmm_k3.fit_predict(X3_sample)

df_sample_k3 = df_clean.iloc[sample_idx3].copy()
df_sample_k3['cluster'] = cluster_labels_k3

print("클러스터별 손잡이 분포:")
print(pd.crosstab(df_sample_k3['cluster'], df_sample_k3['p_throws']))

print("\n클러스터별 구종 분포:")
print(pd.crosstab(df_sample_k3['cluster'], df_sample_k3['pitch_type']))

print("\n클러스터별 평균 특성:")
print(df_sample_k3.groupby('cluster')[features_v3].mean().round(2))

In [ ]:
years = sorted(df_clean['game_year'].unique())
year_results_v3 = {}

for year in years:
    df_year = df_clean[df_clean['game_year'] == year]
    X_year_v3 = scaler3.transform(df_year[features_v3].values)
    
    np.random.seed(42)
    idx = np.random.choice(len(X_year_v3), size=min(20000, len(X_year_v3)), replace=False)
    X_year_sample_v3 = X_year_v3[idx]
    
    scores = {}
    for k in range(2, 7):
        gmm = GaussianMixture(n_components=k, random_state=42, n_init=1)
        labels = gmm.fit_predict(X_year_sample_v3)
        score = silhouette_score(X_year_sample_v3, labels)
        scores[k] = score
    
    year_results_v3[year] = scores
    print(f"{year}년: {scores}")

In [ ]:
cv_results_v3 = {}

for year in years:
    df_year = df_clean[df_clean['game_year'] == year]
    X_year_v3 = scaler3.transform(df_year[features_v3].values)
    
    np.random.seed(42)
    idx = np.random.choice(len(X_year_v3), size=min(20000, len(X_year_v3)), replace=False)
    X_year_sample_v3 = X_year_v3[idx]
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []
    
    for train_idx, test_idx in kf.split(X_year_sample_v3):
        X_train, X_test = X_year_sample_v3[train_idx], X_year_sample_v3[test_idx]
        
        gmm = GaussianMixture(n_components=3, random_state=42, n_init=1)
        gmm.fit(X_train)
        test_labels = gmm.predict(X_test)
        
        if len(set(test_labels)) > 1:
            score = silhouette_score(X_test, test_labels)
            fold_scores.append(score)
    
    cv_results_v3[year] = fold_scores
    print(f"{year}년 - CV 평균: {np.mean(fold_scores):.4f} (fold별: {[f'{s:.3f}' for s in fold_scores]})")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 기본 한글 폰트
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

In [ ]:


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = {'FF': 'crimson', 'SI': 'royalblue', 'FC': 'darkorange'}

for pitch, color in colors.items():
    mask = df_sample_k3['pitch_type'] == pitch
    axes[0].scatter(df_sample_k3.loc[mask, 'release_speed'],
                     df_sample_k3.loc[mask, 'pfx_x_arm'],
                     s=3, alpha=0.3, c=color, label=pitch)
    axes[1].scatter(df_sample_k3.loc[mask, 'pfx_x_arm'],
                     df_sample_k3.loc[mask, 'pfx_z'],
                     s=3, alpha=0.3, c=color, label=pitch)

axes[0].set_xlabel('release_speed (mph)')
axes[0].set_ylabel('pfx_x_arm (arm-side +)')
axes[0].set_title('구속 vs 수평무브먼트')
axes[0].legend()

axes[1].set_xlabel('pfx_x_arm (arm-side +)')
axes[1].set_ylabel('pfx_z (수직무브먼트)')
axes[1].set_title('수평 vs 수직 무브먼트')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/fastball_cluster_scatter.png', dpi=150)
plt.show()

In [ ]:
# 최종 k=3 클러스터링 결과 저장 (v3 피처 기준)
df_final = df_sample_k3.copy()

output_cols = ['game_year', 'pitcher', 'player_name', 'p_throws', 'pitch_type',
               'release_speed', 'pfx_x', 'pfx_z', 'pfx_x_arm', 
               'release_spin_rate', 'spin_axis', 'spin_axis_arm',
               'release_pos_x', 'release_pos_x_arm', 'release_pos_z', 
               'release_extension', 'cluster']

df_final = df_final[output_cols]
df_final.to_csv('../data/processed/fastball_shape_clusters_k3.csv', index=False)

print(f"저장 완료: {df_final.shape}")
print(df_final.head())

## 결과 요약 (출력 삭제 전 기록)

### 데이터
- 전체 2021–2025 Statcast: 3,846,144행 → FF/SI/FC 필터링: 2,100,398행 → 결측 제거: 2,090,319행
- 실루엣 계산은 속도 때문에 전체 50,000개, 연도별 20,000개 랜덤 샘플 사용 (seed=42)

### 1차 시도 (보정 전 피처) — 손잡이 아티팩트
| k | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|
| 실루엣 | 0.282 | 0.175 | 0.197 | 0.180 | 0.156 |

k=2 클러스터가 투수 손잡이와 거의 일치: 클러스터0 = R 31,101 / L 1, 클러스터1 = L 14,031 / R 4,867
→ 구종이 아니라 좌우완을 나눈 결과. 이 구간의 연도별·교차검증 셀도 보정 전 결과라 참고용.

### 최종 (pfx_x, release_pos_x, spin_axis 암사이드 보정 후)
| k | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|
| 실루엣 | 0.249 | 0.173 | 0.161 | 0.076 | 0.061 |

k=3 클러스터별 구종 분포
| 클러스터 | FC | FF | SI | 해석 |
|---|---|---|---|---|
| 0 | 6,607 | 1,440 | 167 | 커터형 |
| 1 | 340 | 26,254 | 877 | 포심형 |
| 2 | 31 | 1,625 | 12,659 | 싱커형 |

손잡이는 세 클러스터 모두 좌완 약 25–30%로 고르게 섞임 → 아티팩트 해소.

k=3 클러스터별 평균
| 클러스터 | 구속(mph) | pfx_x_arm(ft) | pfx_z(ft) | 회전수 | 회전축 |
|---|---|---|---|---|---|
| 0 (커터) | 89.75 | -0.15 | 0.69 | 2374 | 186° |
| 1 (포심) | 94.25 | 0.63 | 1.37 | 2290 | 213° |
| 2 (싱커) | 93.29 | 1.26 | 0.66 | 2155 | 226° |

### 연도별 k=3 실루엣 / 5-fold CV 평균
| 연도 | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|
| 실루엣 | 0.166 | 0.173 | 0.173 | 0.178 | 0.178 |
| CV 평균 | 0.165 | 0.172 | 0.173 | 0.178 | 0.177 |

모든 연도에서 k=2가 실루엣은 더 높았지만 커터 vs 포심·싱커만 구분하므로, 세 구종을 분리하는 k=3을 채택.

### 저장 파일
- `data/processed/fastball_shape_clusters_k3.csv` (50,000행, 클러스터 라벨 포함)
- `data/processed/fastball_cluster_scatter.png`